# MLOps Pipeline Platform Demo
Demonstrates automated training, drift detection, and model registry workflows.

In [ ]:
import mlflow
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

mlflow.set_tracking_uri('http://localhost:5000')
mlflow.set_experiment('mlops-platform-demo')

np.random.seed(42)
n = 10000
X = pd.DataFrame(np.random.randn(n, 20), columns=[f'feature_{i}' for i in range(20)])
y = (X['feature_0'] + X['feature_1'] + np.random.randn(n) * 0.1 > 0).astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

with mlflow.start_run(run_name='demo_training_run'):
    model = GradientBoostingClassifier(n_estimators=100, max_depth=5)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    f1 = f1_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)
    mlflow.log_metrics({'f1_score': f1, 'auc_roc': auc})
    mlflow.sklearn.log_model(model, 'model')
    print(f'F1 Score: {f1:.4f}')
    print(f'AUC-ROC: {auc:.4f}')
    print(f'Run ID: {mlflow.active_run().info.run_id}')